# Прогнозна система для агро-застосунку: моделі 01, 02, 03, 04, 05, 08, 10

**Протокол (єдиний для всіх моделей):**
* розбиття лише за часом (колонка `split`), без випадкового перемішування;
* для горизонту `h` місяців із train/valid викидаються рядки, чий target «заходить» у наступний період (purge), щоб не було витоку;
* вибір моделі: fit на train → оцінка на valid; чесна оцінка: fit на train+valid → test; фінальний прогноз: fit на всіх розмічених рядках → рядок `predict`;
* кожну модель порівнюємо з бейзлайнами; якщо не б'є їх, це видно в таблицях;
* `post_war` / `post_dam` у train завжди 0, тому в прогнозних моделях 01–03/05 не використовуються (екстраполяція була б вигадкою).

Запуск у Colab: завантажте CSV (01, 02, 03, 04, 05, 08, 10) у `/content` або запустіть комірку 1, і вона запропонує завантажити файли.

In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LogisticRegression, LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier, ExtraTreesRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, f1_score, recall_score,
                             precision_score, average_precision_score, balanced_accuracy_score, precision_recall_curve)
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200, 'display.max_columns', 40)

IN_COLAB = os.path.exists('/content')
DATA_DIR = '/content' if IN_COLAB else '/mnt/user-data/uploads'
OUT_DIR = '/content/outputs' if IN_COLAB else '/mnt/user-data/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
if IN_COLAB and not glob.glob(f'{DATA_DIR}/01_ndvi_forecast.csv'):
    from google.colab import files
    files.upload()          # виберіть усі CSV
def P(name): return f'{DATA_DIR}/{name}.csv'
SEED = 0
FC_ROWS = []   # зведена таблиця прогнозів (довгий формат)
METRICS = []   # зведена таблиця метрик

In [ ]:
def months_add(s, h): return (s.dt.to_period('M') + h).dt.to_timestamp()

def make_reg(name):
    if name == 'ridge':
        return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), RidgeCV(alphas=np.logspace(1, 3.5, 15)))
    if name == 'hgb':
        return HistGradientBoostingRegressor(max_depth=3, learning_rate=0.04, max_iter=150, min_samples_leaf=25,
                                             l2_regularization=2.0, random_state=SEED)
    if name == 'et':
        return make_pipeline(SimpleImputer(strategy='median'),
                             ExtraTreesRegressor(300, min_samples_leaf=8, max_features=0.5, n_jobs=-1, random_state=SEED))
REG = ['ridge', 'hgb', 'et']

def fit_predict_reg(Xtr, ytr, Xte_list):
    """Повертає список словників {модель: прогноз} по кожному Xte."""
    outs = [dict() for _ in Xte_list]
    for n in REG:
        m = make_reg(n).fit(Xtr, ytr)
        for o, X in zip(outs, Xte_list): o[n] = m.predict(X)
    for o in outs: o['ensemble'] = np.mean([o[n] for n in REG], axis=0)
    return outs

def reg_scores(y, p):
    return dict(MAE=mean_absolute_error(y, p), RMSE=mean_squared_error(y, p) ** .5, R2=r2_score(y, p))

def baseline_preds(tr, te, tgt, lin_col, persistence):
    out = {'zero': np.zeros(len(te))}
    if persistence: out['persistence'] = te[lin_col].fillna(0).values
    m = make_pipeline(SimpleImputer(strategy='median'), LinearRegression()).fit(tr[[lin_col]], tr[tgt])
    out['lin_1feature'] = m.predict(te[[lin_col]])
    return out

## Модель 01: прогноз NDVI z-score (t+1, t+3, t+6) і модель 03: вологість ґрунту (t+1, t+3)
Спільна функція для регресійних задач. Ознаки: усі погодні/кліматичні/сезонні + `ndvi_z_lag1/2`; без `post_war`, `post_dam`.

In [ ]:
def run_reg(df, feats, stem, horizons, lin_col, persistence, tag):
    vs = df.loc[df.split == 'valid', 'date'].min(); ts = df.loc[df.split == 'test', 'date'].min()
    metrics, store, fc = [], {}, []
    for h in horizons:
        tgt = f'{stem}_t+{h}'; lab = df[tgt].notna(); endt = months_add(df.date, h)
        tr = df[(df.split == 'train') & lab & (endt < vs)]
        va = df[(df.split == 'valid') & lab]
        tv = df[df.split.isin(['train', 'valid']) & lab & (endt < ts)]
        te = df[(df.split == 'test') & lab]
        al = df[df.split.isin(['train', 'valid', 'test']) & lab]
        pr = df[df.split == 'predict']
        Pva, = fit_predict_reg(tr[feats], tr[tgt], [va[feats]])
        Pte, = fit_predict_reg(tv[feats], tv[tgt], [te[feats]])
        Pva.update(baseline_preds(tr, va, tgt, lin_col, persistence))
        Pte.update(baseline_preds(tv, te, tgt, lin_col, persistence))
        for name in Pva:
            metrics.append(dict(set=tag, horizon=h, model=name, split='valid', n=len(va), **reg_scores(va[tgt], Pva[name])))
            metrics.append(dict(set=tag, horizon=h, model=name, split='test', n=len(te), **reg_scores(te[tgt], Pte[name])))
        cand = REG + ['ensemble', 'lin_1feature']
        best = min(cand, key=lambda n: mean_absolute_error(va[tgt], Pva[n]))
        res = te[tgt] - Pte[best]; q10, q90 = np.quantile(res, [.1, .9])
        # фінальна модель на всіх розмічених даних
        if len(pr):
            outs, = fit_predict_reg(al[feats], al[tgt], [pr[feats]])
            outs['lin_1feature'] = make_pipeline(SimpleImputer(strategy='median'), LinearRegression()).fit(al[[lin_col]], al[tgt]).predict(pr[[lin_col]])
            zva = va[tgt].abs().mean(); zte = te[tgt].abs().mean()
            for (i, r), p in zip(pr.iterrows(), outs[best]):
                fc.append(dict(set=tag, zone=r.zone, origin=r.date, horizon=h, target_date=(r.date.to_period('M') + h).to_timestamp(),
                               target=tgt, value=p, p10=p + q10, p90=p + q90, model=best,
                               test_MAE=mean_absolute_error(te[tgt], Pte[best]),
                               valid_skill_vs_zero=1 - mean_absolute_error(va[tgt], Pva[best]) / zva,
                               test_skill_vs_zero=1 - mean_absolute_error(te[tgt], Pte[best]) / zte))
        store[h] = dict(best=best, va=pd.Series(Pva[best], index=va.index), te=pd.Series(Pte[best], index=te.index),
                        te_all=Pte, te_df=te, tgt=tgt)
    return pd.DataFrame(metrics), pd.DataFrame(fc), store

d1 = pd.read_csv(P('01_ndvi_forecast'), parse_dates=['date'])
EXC = ['zone', 'country', 'zone_type', 'date', 'split', 'post_war', 'post_dam']
F1_ = [c for c in d1.columns if c not in EXC and not c.startswith('target')]
print(len(F1_), 'ознак:', F1_)
m01, fc01, st01 = run_reg(d1, F1_, 'target_ndvi_z', [1, 3, 6], 'ndvi_z_lag1', True, '01_ndvi')
print(m01.pivot_table(index=['horizon', 'model'], columns='split', values=['MAE', 'RMSE', 'R2']).round(3))

In [ ]:
print('Найкраща модель по горизонтах:', {h: v['best'] for h, v in st01.items()})
# похибка по зонах на test (найкраща модель) для чесного погляду на зсув війни/дамби
for h, v in st01.items():
    t = v['te_df'].assign(err=(v['te_df'][v['tgt']] - v['te']).abs())
    print('t+%d MAE test по зонах:' % h, t.groupby('zone').err.mean().round(2).to_dict())
print(fc01.round(3))
METRICS.append(m01); FC_ROWS.append(fc01)

In [ ]:
d3 = pd.read_csv(P('03_soil_moisture_warning'), parse_dates=['date'])
F3_ = [c for c in d3.columns if c not in EXC and not c.startswith('target')]
m03, fc03, st03 = run_reg(d3, F3_, 'target_sm_root_z', [1, 3], 'sm_root_mm_anom', False, '03_soil_moisture')
print(m03.pivot_table(index=['horizon', 'model'], columns='split', values=['MAE', 'RMSE', 'R2']).round(3))
print(fc03.round(3))
METRICS.append(m03); FC_ROWS.append(fc03)

## Модель 02: світлофор посухи (red / yellow / green)
Класи виведені з NDVI z-score (red < −1, yellow −1…−0.5). Порівнюємо: мажоритарний клас, persistence-правило з `ndvi_z_lag1`,
логістичну регресію та HGB з `class_weight='balanced'`, а також «регресія 01 + пороги». Метрики: macro-F1, recall для red.

In [ ]:
CL = ['red_drought', 'yellow_risk', 'green_normal']; CMAP = {c: i for i, c in enumerate(CL)}
d2 = pd.read_csv(P('02_drought_traffic_light'), parse_dates=['date'])
F2_ = [c for c in d2.columns if c not in EXC and not c.startswith('target')]
# перевірка узгодженості порогів із target 01
chk = d1['target_ndvi_z_t+1'].dropna().index
cls_from_z = np.where(d1.loc[chk, 'target_ndvi_z_t+1'] < -1, 'red_drought', np.where(d1.loc[chk, 'target_ndvi_z_t+1'] < -.5, 'yellow_risk', 'green_normal'))
print('Збіг класів 02 з порогами по target 01 (t+1):', (cls_from_z == d2.loc[chk, 'target_drought_class_t+1']).mean().round(4))

def make_clf(name):
    if name == 'logreg':
        return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(C=.2, class_weight='balanced', max_iter=2000))
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=120, min_samples_leaf=20,
                                          l2_regularization=2.0, class_weight='balanced', random_state=SEED)

def z_to_cls(z): return np.where(z < -1, 0, np.where(z < -.5, 1, 2))
def clf_scores(y, pred):
    return dict(macroF1=f1_score(y, pred, average='macro'), recall_red=recall_score(y, pred, labels=[0], average='macro', zero_division=0),
                precision_red=precision_score(y, pred, labels=[0], average='macro', zero_division=0), bal_acc=balanced_accuracy_score(y, pred))

vs = d2.loc[d2.split == 'valid', 'date'].min(); ts = d2.loc[d2.split == 'test', 'date'].min()
metrics, fc, st02 = [], [], {}
for h in [1, 3, 6]:
    tgt = f'target_drought_class_t+{h}'; y = d2[tgt].map(CMAP); lab = y.notna(); endt = months_add(d2.date, h)
    tr = d2[(d2.split == 'train') & lab & (endt < vs)]; va = d2[(d2.split == 'valid') & lab]
    tv = d2[d2.split.isin(['train', 'valid']) & lab & (endt < ts)]; te = d2[(d2.split == 'test') & lab]
    al = d2[d2.split.isin(['train', 'valid', 'test']) & lab]; pr = d2[d2.split == 'predict']
    pv, pt = {}, {}
    for n in ['logreg', 'hgb']:
        pv[n] = make_clf(n).fit(tr[F2_], y[tr.index]).predict_proba(va[F2_])
        pt[n] = make_clf(n).fit(tv[F2_], y[tv.index]).predict_proba(te[F2_])
    lab_v = {n: p.argmax(1) for n, p in pv.items()}; lab_t = {n: p.argmax(1) for n, p in pt.items()}
    for L, dd in ((lab_v, va), (lab_t, te)):
        L['majority_green'] = np.full(len(dd), 2); L['persistence_rule'] = z_to_cls(dd['ndvi_z_lag1'].fillna(0).values)
    lab_v['reg01+thresholds'] = z_to_cls(st01[h]['va'].reindex(va.index).values)
    lab_t['reg01+thresholds'] = z_to_cls(st01[h]['te'].reindex(te.index).values)
    for n in lab_v:
        metrics.append(dict(set='02_drought', horizon=h, model=n, split='valid', n=len(va), **clf_scores(y[va.index], lab_v[n])))
        metrics.append(dict(set='02_drought', horizon=h, model=n, split='test', n=len(te), **clf_scores(y[te.index], lab_t[n])))
    best = max(['logreg', 'hgb', 'reg01+thresholds'], key=lambda n: f1_score(y[va.index], lab_v[n], average='macro'))
    prob_model = max(['logreg', 'hgb'], key=lambda n: f1_score(y[va.index], lab_v[n], average='macro'))
    m = make_clf(prob_model).fit(al[F2_], y[al.index]); pp = m.predict_proba(pr[F2_])
    reg_val = fc01[fc01.horizon == h].set_index('zone').value
    for (i, r), p in zip(pr.iterrows(), pp):
        lbl = CL[int(z_to_cls(np.array([reg_val[r.zone]]))[0])] if best == 'reg01+thresholds' else CL[int(p.argmax())]
        fc.append(dict(set='02_drought', zone=r.zone, origin=r.date, horizon=h, target_date=(r.date.to_period('M') + h).to_timestamp(),
                       target=tgt, label=lbl, prob_red=p[0], prob_yellow=p[1], prob_green=p[2],
                       model=best if best != 'reg01+thresholds' else f'reg01+thresholds (probabilities: {prob_model})',
                       test_macroF1=f1_score(y[te.index], lab_t[best], average='macro'),
                       test_macroF1_persistence=f1_score(y[te.index], lab_t['persistence_rule'], average='macro'),
                       test_recall_red=recall_score(y[te.index], lab_t[best], labels=[0], average='macro', zero_division=0)))
m02, fc02 = pd.DataFrame(metrics), pd.DataFrame(fc)
print(m02.pivot_table(index=['horizon', 'model'], columns='split', values=['macroF1', 'recall_red']).round(3))
print(fc02.round(3))
METRICS.append(m02); FC_ROWS.append(fc02)

## Модель 05: чи допомагають 7 ознак екстремальних подій (дані з 2015)
Чесне порівняння на **тих самих** рядках: (A) ознаки з 01–03 і (B) ті ж + 7 екстремальних, обидві навчені лише на train 05 (2015+).
Додатково (C): модель 01–03, навчена на повній історії 2003–2017, оцінена на тих самих valid/test рядках 05.
**Увага:** у файлі 05 немає рядка `predict` (грудень 2025), тому ця модель дає порівняння якості, а не прогноз.

In [ ]:
d5 = pd.read_csv(P('05_ndvi_forecast_with_extremes_2015plus'), parse_dates=['date'])
EXT = [c for c in d5.columns if c.endswith('_anom') and c in ['hot_days_30_anom', 'very_hot_days_35_anom', 'frost_days_anom',
       'max_dry_spell_days_anom', 'dry_days_anom', 'tmax_peak_c_anom', 'max_rain_intensity_mm_h_anom']]
BASE5 = [c for c in F1_ if c in d5.columns]
print('екстремальні ознаки:', EXT)
rows5 = []
vs5 = d5.loc[d5.split == 'valid', 'date'].min(); ts5 = d5.loc[d5.split == 'test', 'date'].min()
def reg_cfg(stem, h): return f'{stem}_t+{h}'
for stem, hs in [('target_ndvi_z', [1, 3, 6]), ('target_sm_root_z', [1, 3])]:
    for h in hs:
        tgt = reg_cfg(stem, h); lab = d5[tgt].notna(); endt = months_add(d5.date, h)
        tr = d5[(d5.split == 'train') & lab & (endt < vs5)]; va = d5[(d5.split == 'valid') & lab]
        tv = d5[d5.split.isin(['train', 'valid']) & lab & (endt < ts5)]; te = d5[(d5.split == 'test') & lab]
        full = d1 if stem == 'target_ndvi_z' else d3
        fl = full[(full.split == 'train') & full[tgt].notna() & (months_add(full.date, h) < vs5)]
        fl_tv = full[full.split.isin(['train', 'valid']) & full[tgt].notna() & (months_add(full.date, h) < ts5)]
        for feats, nm in [(BASE5, 'A_base_2015+'), (BASE5 + EXT, 'B_base+extremes_2015+')]:
            pv, = fit_predict_reg(tr[feats], tr[tgt], [va[feats]]); pt, = fit_predict_reg(tv[feats], tv[tgt], [te[feats]])
            for mn in REG + ['ensemble']:
                rows5.append(dict(target=tgt, variant=nm, model=mn, valid_MAE=mean_absolute_error(va[tgt], pv[mn]), test_MAE=mean_absolute_error(te[tgt], pt[mn])))
        pv, = fit_predict_reg(fl[BASE5], fl[tgt], [va[BASE5]]); pt, = fit_predict_reg(fl_tv[BASE5], fl_tv[tgt], [te[BASE5]])
        for mn in REG + ['ensemble']:
            rows5.append(dict(target=tgt, variant='C_base_full_2003+', model=mn, valid_MAE=mean_absolute_error(va[tgt], pv[mn]), test_MAE=mean_absolute_error(te[tgt], pt[mn])))
        rows5.append(dict(target=tgt, variant='zero_baseline', model='zero', valid_MAE=va[tgt].abs().mean(), test_MAE=te[tgt].abs().mean()))
m05 = pd.DataFrame(rows5)
# найкраща модель кожного варіанта за valid MAE → її test MAE
sel = m05[m05.variant != 'zero_baseline'].sort_values('valid_MAE').groupby(['target', 'variant']).head(1)
print(sel.pivot(index='target', columns='variant', values='test_MAE').join(m05[m05.variant == 'zero_baseline'].set_index('target').test_MAE.rename('zero_baseline')).round(3))
print('\nУсереднено по всіх моделях/горизонтах (test MAE):'); print(m05.groupby('variant').test_MAE.mean().round(3))

## Модель 04: «очікуване проти фактичного», ефект людини
Навчаємо **лише на погоді** (без лагів NDVI і без `post_war/post_dam`) на `train_pre2022`; прогноз на `apply_2022_2025` = «що було б без людини».
Залишок = факт − очікуване. Значущість: moving-block bootstrap (блок 6 міс.) через автокореляцію; ефект дамби додатково як
різниця-різниць (UA_main/UA_second мінус UA_control) між періодами «війна» та «війна+дамба».

In [ ]:
d4 = pd.read_csv(P('04_expected_vs_actual_human_effect'), parse_dates=['date'])
F4_ = [c for c in d4.columns if c not in EXC + ['target_ndvi_z_t0']]
T4 = 'target_ndvi_z_t0'
pre = d4[(d4.split == 'train_pre2022') & d4[T4].notna()]
fit_a = pre[pre.date < '2018-01-01']; val_a = pre[pre.date >= '2018-01-01']
Pv, = fit_predict_reg(fit_a[F4_], fit_a[T4], [val_a[F4_]])
rows = [dict(model='zero', **reg_scores(val_a[T4], np.zeros(len(val_a))))] + [dict(model=n, **reg_scores(val_a[T4], p)) for n, p in Pv.items()]
print('Valid 2018–2021 (навчання 2003–2017):'); print(pd.DataFrame(rows).round(3))
best4 = min(REG + ['ensemble'], key=lambda n: mean_squared_error(val_a[T4], Pv[n]))
print('Обрана модель:', best4)
app = d4[d4.split == 'apply_2022_2025'].copy()
Pa, = fit_predict_reg(pre[F4_], pre[T4], [app[F4_]])
app['expected'] = Pa[best4]; app['residual'] = app[T4] - app['expected']
val_a = val_a.assign(residual=val_a[T4] - Pv[best4])
print('Залишки на valid (нуль-гіпотеза шуму), середнє по зонах:', val_a.groupby('zone').residual.mean().round(2).to_dict())

In [ ]:
def block_ci(x, block=6, B=3000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]; n = len(x)
    if n < 3: return (np.nan, np.nan)
    b = min(block, n); nb = int(np.ceil(n / b)); rng = np.random.default_rng(seed)
    st = rng.integers(0, n - b + 1, size=(B, nb)); idx = (st[:, :, None] + np.arange(b)).reshape(B, -1)[:, :n]
    return tuple(np.percentile(x[idx].mean(1), [2.5, 97.5]))
def block_diff_ci(a, c, block=6, B=3000, seed=1):
    a = np.asarray(a, float); c = np.asarray(c, float)
    def bs(x, rng):
        n = len(x); b = min(block, n); nb = int(np.ceil(n / b)); st = rng.integers(0, n - b + 1, size=(B, nb))
        return x[(st[:, :, None] + np.arange(b)).reshape(B, -1)[:, :n]].mean(1)
    rng = np.random.default_rng(seed); d = bs(a, rng) - bs(c, rng); return tuple(np.percentile(d, [2.5, 97.5]))
def phase(r):
    if r.date < pd.Timestamp('2022-03-01'): return '0_pre_war_2022'
    if r.zone in ('UA_main', 'UA_second') and r.date >= pd.Timestamp('2023-07-01'): return '2_war+dam'
    if r.zone.startswith('UA'): return '1_war' if r.date < pd.Timestamp('2023-07-01') else '2_war+dam'
    return '3_AU_all_2022-25'
app['phase'] = app.apply(phase, axis=1)
tab = []
for (z, ph), g in app.groupby(['zone', 'phase']):
    lo, hi = block_ci(g.residual); tab.append(dict(zone=z, phase=ph, n=g.residual.notna().sum(), mean_residual=g.residual.mean(), ci95_low=lo, ci95_high=hi,
                       significant=(lo > 0) or (hi < 0), valid_mean_residual=val_a[val_a.zone == z].residual.mean()))
tab = pd.DataFrame(tab).round(3); print(tab)
# різниця-різниць: ефект дамби = (UA_main − UA_control) у період war+dam мінус у період war
piv = app.pivot(index='date', columns='zone', values='residual')
dd_rows = []
for z in ['UA_main', 'UA_second']:
    diff = piv[z] - piv['UA_control']
    a = diff[(diff.index >= '2023-07-01')].dropna(); c = diff[(diff.index >= '2022-03-01') & (diff.index < '2023-07-01')].dropna()
    lo, hi = block_diff_ci(a, c); dd_rows.append(dict(zone=z, dam_effect_DiD=a.mean() - c.mean(), ci95_low=lo, ci95_high=hi, n_dam=len(a), n_war=len(c)))
print('\nЕфект дамби (різниця-різниць відносно UA_control):'); print(pd.DataFrame(dd_rows).round(3))
fig, axs = plt.subplots(5, 1, figsize=(11, 11), sharex=True)
for ax, z in zip(axs, piv.columns):
    ax.plot(piv.index, piv[z]); ax.axhline(0, c='k', lw=.6); ax.axvline(pd.Timestamp('2022-03-01'), c='r', ls='--'); ax.axvline(pd.Timestamp('2023-07-01'), c='b', ls='--'); ax.set_ylabel(z)
axs[0].set_title('Залишок NDVI z (факт − очікуване без людини); червоне = початок війни, синє = дамба'); plt.tight_layout(); plt.show()
app[['zone', 'date', T4, 'expected', 'residual', 'phase']].to_csv(f'{OUT_DIR}/human_effect_residuals.csv', index=False)
tab.to_csv(f'{OUT_DIR}/human_effect_summary.csv', index=False)

## Модель 08: сповіщення на 8 днів (low_rain, dry_spell, heat, frost)
Рідкісні події, тому: `class_weight='balanced'`, метрики PR-AUC / F1 / recall. Поріг підбирається **не на одному valid**
(там 3–7 позитивів), а за pooled out-of-fold прогнозами розширюваного вікна 2018–2023; потім чесна оцінка на test.
Бейзлайни: persistence (поточний прапор → наступний) та сезонна частота події по зоні.

In [ ]:
d8 = pd.read_csv(P('08_alerts_8day'), parse_dates=['period_start'])
d8['year'] = d8.period_start.dt.year
d8['poy_sin'] = np.sin(2 * np.pi * d8.period_of_year / 46); d8['poy_cos'] = np.cos(2 * np.pi * d8.period_of_year / 46)
Z = pd.get_dummies(d8.zone, prefix='z').astype(float); d8 = pd.concat([d8, Z], axis=1)
F8 = [c for c in d8.columns if c not in ['zone', 'period_start', 'split', 'year', 'period_days'] and not c.startswith('target')]
ALERTS = ['low_rain', 'dry_spell', 'heat', 'frost']
def make_clf8(name):
    if name == 'logreg':
        return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(C=.1, class_weight='balanced', max_iter=3000))
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=100, min_samples_leaf=30, l2_regularization=3.0,
                                          class_weight='balanced', random_state=SEED)
def best_thr(y, p):
    pr_, rc_, th = precision_recall_curve(y, p); f = 2 * pr_[:-1] * rc_[:-1] / np.maximum(pr_[:-1] + rc_[:-1], 1e-9)
    return th[int(np.nanargmax(f))]
def cls_metrics(y, p, thr):
    yh = (p >= thr).astype(int)
    return dict(n_pos=int(y.sum()), prevalence=y.mean(), PR_AUC=average_precision_score(y, p) if y.sum() else np.nan,
                F1=f1_score(y, yh, zero_division=0), recall=recall_score(y, yh, zero_division=0), precision=precision_score(y, yh, zero_division=0), threshold=thr)
rows8, fc8 = [], []
for a in ALERTS:
    tgt = f'target_alert_{a}_next8d'; lab = d8[tgt].notna()
    dev = d8[d8.split.isin(['train', 'valid']) & lab]; te = d8[(d8.split == 'test') & lab]; al = d8[d8.split.isin(['train', 'valid', 'test']) & lab]; pr = d8[d8.split == 'predict']
    oof = {n: [] for n in ['logreg', 'hgb']}; yoof = []
    for yr in range(2018, 2024):
        trn, tst = dev[dev.year < yr], dev[dev.year == yr]
        if trn[tgt].nunique() < 2: continue
        yoof.append(tst[tgt].values)
        for n in oof: oof[n].append(make_clf8(n).fit(trn[F8], trn[tgt]).predict_proba(tst[F8])[:, 1])
    yoof = np.concatenate(yoof); oof = {n: np.concatenate(v) for n, v in oof.items()}
    ap = {n: average_precision_score(yoof, p) for n, p in oof.items()}; best = max(ap, key=ap.get); thr = best_thr(yoof, oof[best])
    for n in oof: rows8.append(dict(alert=a, model=n, split='OOF_2018-23', **cls_metrics(yoof, oof[n], best_thr(yoof, oof[n]))))
    ptest = {n: make_clf8(n).fit(dev[F8], dev[tgt]).predict_proba(te[F8])[:, 1] for n in oof}
    for n in oof: rows8.append(dict(alert=a, model=n, split='test', **cls_metrics(te[tgt].values, ptest[n], best_thr(yoof, oof[n]))))
    rows8.append(dict(alert=a, model='persistence', split='test', **cls_metrics(te[tgt].values, te[f'alert_{a}'].values.astype(float), .5)))
    seas = dev.groupby(['zone', 'period_of_year'])[tgt].mean(); sc = te.set_index(['zone', 'period_of_year']).index.map(lambda k: seas.get(k, dev[tgt].mean()))
    rows8.append(dict(alert=a, model='seasonal_rate', split='test', n_pos=int(te[tgt].sum()), prevalence=te[tgt].mean(), PR_AUC=average_precision_score(te[tgt], np.array(sc, float))))
    m = make_clf8(best).fit(al[F8], al[tgt]); pp = m.predict_proba(pr[F8])[:, 1]
    tm = cls_metrics(te[tgt].values, ptest[best], thr)
    # «надійність»: PR-AUC на test ≥ 2× частоти події, кращий за persistence і сезонну частоту, і ≥10 позитивів у test
    reliable = bool(tm['PR_AUC'] >= 2 * tm['prevalence'] and tm['PR_AUC'] > average_precision_score(te[tgt], te[f'alert_{a}'].astype(float))
                    and tm['PR_AUC'] > average_precision_score(te[tgt], np.array(sc, float)) and tm['n_pos'] >= 10 and tm['F1'] > 0.2)
    for (i, r), p in zip(pr.iterrows(), pp):
        fc8.append(dict(set='08_alerts', zone=r.zone, origin=r.period_start, horizon='next 8d', target=tgt, value=p,
                        label=(int(p >= thr) if reliable else np.nan), reliable=reliable, model=best, threshold=thr,
                        test_PR_AUC=average_precision_score(te[tgt], ptest[best]), test_recall=cls_metrics(te[tgt].values, ptest[best], thr)['recall'],
                        test_F1=cls_metrics(te[tgt].values, ptest[best], thr)['F1']))
m08, fc08 = pd.DataFrame(rows8), pd.DataFrame(fc8)
print(m08.round(3).to_string()); print(fc08.round(3))
METRICS.append(m08.assign(set='08_alerts')); FC_ROWS.append(fc08)

## Модель 10: NDVI → урожайність зернових (по країнах, щороку)
46 рядків, 2 країни, 23 роки; 2025 без target, це рядок прогнозу. Валідація **leave-one-year-out** (виключається рік в обох країнах).
Дуже мала вибірка, тому лише прості моделі, бейзлайни: середнє по країні та лінійний тренд по країні.

In [ ]:
d10 = pd.read_csv(P('10_harvest_yield_link')); T10 = 'target_cereal_yield_kg_ha'
lab10 = d10[d10[T10].notna()].reset_index(drop=True); pr10 = d10[d10[T10].isna()].reset_index(drop=True)
ENV = ['ndvi', 'precip_mm', 'sm_root_mm', 'tair_c', 'vpd_kpa', 'et_mm']
def X_of(df, cols, year):
    X = df[cols].copy(); X['ua'] = (df.country == 'ukraine').astype(float)
    if year: X['year'] = df.year - 2003
    return X
def m_country_mean(tr, te): return te.country.map(tr.groupby('country')[T10].mean()).values
def m_trend(tr, te):
    out = np.zeros(len(te))
    for c in te.country.unique():
        g = tr[tr.country == c]; k = np.polyfit(g.year, g[T10], 1); out[(te.country == c).values] = np.polyval(k, te.loc[te.country == c, 'year'])
    return out
def m_ridge(cols, year):
    def f(tr, te):
        m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 2, 20))).fit(X_of(tr, cols, year), tr[T10]); return m.predict(X_of(te, cols, year))
    return f
def m_percountry(cols):
    def f(tr, te):
        out = np.zeros(len(te))
        for c in te.country.unique():
            g = tr[tr.country == c]; m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 2, 15))).fit(g[cols + ['year']], g[T10])
            out[(te.country == c).values] = m.predict(te.loc[te.country == c, cols + ['year']])
        return out
    return f
M10 = {'country_mean': m_country_mean, 'country_trend': m_trend, 'ridge_ndvi+year': m_ridge(['ndvi'], True), 'ridge_ndvi_only': m_ridge(['ndvi'], False),
       'ridge_env+year': m_ridge(ENV, True), 'per_country_ridge_ndvi+year': m_percountry(['ndvi']), 'per_country_ridge_env+year': m_percountry(ENV)}
loyo = {n: np.zeros(len(lab10)) for n in M10}
for yr in sorted(lab10.year.unique()):
    te_i = lab10.index[lab10.year == yr]; tr = lab10[lab10.year != yr]; te = lab10.loc[te_i]
    for n, f in M10.items(): loyo[n][te_i] = f(tr, te)
r10 = []
for n, p in loyo.items():
    r10.append(dict(model=n, **reg_scores(lab10[T10], p), MAE_australia=mean_absolute_error(lab10[T10][lab10.country == 'australia'], p[lab10.country == 'australia']),
                    MAE_ukraine=mean_absolute_error(lab10[T10][lab10.country == 'ukraine'], p[lab10.country == 'ukraine'])))
m10 = pd.DataFrame(r10).sort_values('MAE').round(3); print(m10)
print('\nКореляція ndvi↔yield по країнах:', lab10.groupby('country').apply(lambda g: g.ndvi.corr(g[T10])).round(2).to_dict())
best10 = m10.model.iloc[0]; res = lab10[T10] - loyo[best10]; q10_, q90_ = np.quantile(res, [.1, .9])
pp = M10[best10](lab10, pr10)
fc10 = pd.DataFrame([dict(set='10_yield', zone=r.country, origin=pd.Timestamp(f'{int(r.year)}-12-01'), horizon='year %d' % r.year, target=T10, value=p, p10=p + q10_, p90=p + q90_,
                          model=best10, test_MAE=m10.MAE.iloc[0]) for (i, r), p in zip(pr10.iterrows(), pp)])
print(fc10.round(1)); METRICS.append(m10.assign(set='10_yield')); FC_ROWS.append(fc10)

## Збірка результатів

In [ ]:
fc_all = pd.concat(FC_ROWS, ignore_index=True)
cols_first = ['set', 'zone', 'origin', 'horizon', 'target_date', 'target', 'value', 'p10', 'p90', 'label', 'prob_red', 'prob_yellow', 'prob_green', 'model']
fc_all = fc_all[cols_first + [c for c in fc_all.columns if c not in cols_first]]
fc_all.to_csv(f'{OUT_DIR}/predictions.csv', index=False)
pd.concat(METRICS, ignore_index=True).to_csv(f'{OUT_DIR}/metrics.csv', index=False)
m05.to_csv(f'{OUT_DIR}/metrics_05_extremes_comparison.csv', index=False)
print(fc_all.shape); print(fc_all.round(3).to_string())

# Частина 2. Аналітичні набори (без прогнозних моделей): 06, 07, 09, 11
Спільні інструменти: тест Манна-Кендалла (з поправкою на однакові значення), нахил Сена з 95% ДІ, виправлення на автокореляцію,
корекція множинних порівнянь (Benjamini–Hochberg FDR) і moving-block bootstrap для рядів із пам'яттю.

In [ ]:
from scipy import stats

def mk_test(x):
    """Манн-Кендалл + нахил Сена (на крок індексу) + 95% ДІ нахилу. NaN ігноруються (індекс = позиція в оригіналі)."""
    x = np.asarray(x, float); t = np.arange(len(x)); ok = ~np.isnan(x); x, t = x[ok], t[ok]; n = len(x)
    if n < 8: return dict(n=n, tau=np.nan, p=np.nan, sen=np.nan, sen_lo=np.nan, sen_hi=np.nan)
    diff = x[None, :] - x[:, None]; iu = np.triu_indices(n, 1); S = np.sign(diff[iu]).sum()
    _, cnt = np.unique(x, return_counts=True); tie = (cnt * (cnt - 1) * (2 * cnt + 5)).sum()
    var = (n * (n - 1) * (2 * n + 5) - tie) / 18; z = 0 if S == 0 else (S - np.sign(S)) / np.sqrt(var)
    slopes = (diff[iu] / (t[None, :] - t[:, None])[iu]); sen = np.median(slopes)
    N = len(slopes); C = stats.norm.ppf(.975) * np.sqrt(var); lo, hi = int(np.floor((N - C) / 2)), int(np.ceil((N + C) / 2))
    ss = np.sort(slopes); return dict(n=n, tau=S / (n * (n - 1) / 2), p=2 * (1 - stats.norm.cdf(abs(z))), sen=sen,
                                     sen_lo=ss[max(lo, 0)], sen_hi=ss[min(hi, N - 1)])

def mk_tfpw(x):
    """Trend-free pre-whitening (Yue et al. 2002): p-значення MK після прибирання AR(1) автокореляції."""
    x = np.asarray(x, float); r = mk_test(x); t = np.arange(len(x)); d = x - r['sen'] * t
    d = d - np.nanmean(d); r1 = pd.Series(d).autocorr(1)
    if not np.isfinite(r1) or abs(r1) < .1: return r['p'], r1
    y = (d[1:] - r1 * d[:-1]) + r['sen'] * t[1:]; return mk_test(y)['p'], r1

def bh_fdr(p):
    p = np.asarray(p, float); out = np.full(len(p), np.nan); ok = ~np.isnan(p); pv = p[ok]; n = len(pv)
    o = np.argsort(pv); adj = pv[o] * n / (np.arange(n) + 1); adj = np.minimum.accumulate(adj[::-1])[::-1]; res = np.empty(n); res[o] = np.minimum(adj, 1)
    out[ok] = res; return out

def block_idx(n, block, B, rng):
    b = min(block, n); nb = int(np.ceil(n / b)); st = rng.integers(0, n - b + 1, size=(B, nb))
    return (st[:, :, None] + np.arange(b)).reshape(B, -1)[:, :n]
def boot_mean_ci(x, block=6, B=3000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 4: return (np.nan, np.nan)
    return tuple(np.percentile(x[block_idx(len(x), block, B, np.random.default_rng(seed))].mean(1), [2.5, 97.5]))
def boot_diff_ci(a, c, block=6, B=3000, seed=1):
    a = np.asarray(a, float); a = a[~np.isnan(a)]; c = np.asarray(c, float); c = c[~np.isnan(c)]; rng = np.random.default_rng(seed)
    if len(a) < 4 or len(c) < 4: return (np.nan, np.nan)
    d = a[block_idx(len(a), block, B, rng)].mean(1) - c[block_idx(len(c), block, B, rng)].mean(1); return tuple(np.percentile(d, [2.5, 97.5]))
def corr_neff(x, y):
    """Pearson r і p з ефективною кількістю степенів свободи (Bretherton 1999) для автокорельованих рядів."""
    ok = ~(np.isnan(x) | np.isnan(y)); x, y = x[ok], y[ok]; n = len(x)
    if n < 12 or x.std() == 0 or y.std() == 0: return np.nan, np.nan, n, np.nan
    r = np.corrcoef(x, y)[0, 1]; rx, ry = pd.Series(x).autocorr(1), pd.Series(y).autocorr(1)
    ne = max(n * (1 - rx * ry) / (1 + rx * ry), 4.0) if np.isfinite(rx * ry) else n; ne = min(ne, n)
    tt = r * np.sqrt((ne - 2) / max(1 - r * r, 1e-12)); return r, 2 * (1 - stats.t.cdf(abs(tt), ne - 2)), n, ne

## Файл 06: тренди вегетаційного сезону (Манн-Кендалл + нахил Сена)
5 зон × 10 змінних × 23 роки (2003–2025; у кожному році по 5 місяців сезону). Для кожного тесту: нахил Сена на рік і на десятиліття, % від середнього,
p за Манном-Кендаллом, p із pre-whitening (захист від автокореляції), FDR по всіх 50 тестах. **Бейдж:** ↑/↓ + рівень певності.

In [ ]:
d6 = pd.read_csv(P('06_trends_growing_season_yearly')).sort_values(['zone', 'year'])
V6 = ['ndvi', 'precip_mm', 'sm_root_mm', 'et_mm', 'tair_c', 'vpd_kpa', 'rh_pct', 'runoff_mm', 'ozone_du', 'water_vapour_kg_m2']
rows = []
for z, g in d6.groupby('zone'):
    assert g.year.diff().dropna().eq(1).all()
    for v in V6:
        r = mk_test(g[v].values); p2, r1 = mk_tfpw(g[v].values)
        rows.append(dict(zone=z, variable=v, n_years=r['n'], mean=g[v].mean(), sen_per_year=r['sen'], sen_ci_lo=r['sen_lo'], sen_ci_hi=r['sen_hi'],
                         sen_per_decade=r['sen'] * 10, pct_of_mean_per_decade=100 * r['sen'] * 10 / abs(g[v].mean()) if g[v].mean() else np.nan,
                         tau=r['tau'], p_mk=r['p'], p_prewhitened=p2, lag1_autocorr=r1))
t6 = pd.DataFrame(rows); t6['p_use'] = np.fmax(t6.p_mk, t6.p_prewhitened.fillna(t6.p_mk))  # консервативніше з двох
t6['p_fdr'] = bh_fdr(t6.p_use)
def badge(r):
    a = '↑' if r.sen_per_year > 0 else '↓'
    if r.p_use < .05 and r.p_fdr < .10: return a + ' значущий (FDR)'
    if r.p_use < .05: return a + ' попередньо (p<0.05, без FDR)'
    if r.p_use < .10: return a + ' слабкий (p<0.10)'
    return '— тренду немає'
t6['badge'] = t6.apply(badge, axis=1)
t6.round(4).to_csv(f'{OUT_DIR}/trends_06.csv', index=False)
print(t6.pivot(index='variable', columns='zone', values='badge').to_string())
print('\nСен/десятиліття (значущі, p<0.05):'); print(t6[t6.p_use < .05][['zone', 'variable', 'sen_per_decade', 'pct_of_mean_per_decade', 'p_use', 'p_fdr']].round(4).to_string(index=False))
print(f'\nЗ 50 тестів p<0.05: {(t6.p_use<.05).sum()} (за випадковості очікували б ≈2.5); після FDR<0.10: {(t6.p_fdr<.10).sum()}')

## Файл 07: ланцюгова реакція GHG → океанічні цикли → дощ/атмосфера → ґрунт → поле
Для кожної ланки «причина → наслідок» рахуємо кореляцію наслідку в момент *t* з причиною в *t−k*, k = 0…6 міс., окремо по зонах
(глобальні ряди GHG/ENSO/NAO/SAM однакові в усіх зонах, тому ланки GHG→океан рахуються один раз). p-значення з поправкою на автокореляцію (n_eff) і FDR по всіх тестах.
Кореляція ≠ причинність, а k=0 лише одночасність. Ребро на графі: значуще (FDR<0.05) і з однаковим знаком принаймні в більшості зон країни.

In [ ]:
d7 = pd.read_csv(P('07_chain_reaction_lags'), parse_dates=['date']).sort_values(['zone', 'date']).reset_index(drop=True)
STAGES = {'GHG': ['co2_growth_12m', 'ch4_growth_12m'], 'Океанічні цикли': ['enso_lag0', 'nao_lag0', 'sam_lag0'],
          'Дощ/атмосфера': ['spi3_approx', 'vpd_kpa_anom', 'tair_c_anom', 'water_vapour_kg_m2_anom'],
          'Ґрунт': ['sm_root_mm_anom', 'et_mm_anom'], 'Поле': ['ndvi_z']}
names = list(STAGES)
LINKS = [(c, e) for a, b in zip(names[:-1], names[1:]) for c in STAGES[a] for e in STAGES[b]]
LINKS += [(c, 'ndvi_z') for c in ['spi3_approx', 'vpd_kpa_anom', 'tair_c_anom']]   # «короткі» ланки дощ/атмосфера → поле
LAGS = range(0, 7)
glob_ok = all(np.allclose(d7[d7.zone == z].set_index('date')[['co2_growth_12m', 'enso_lag0']].fillna(0).values,
              d7[d7.zone == 'UA_main'].set_index('date')[['co2_growth_12m', 'enso_lag0']].fillna(0).values) for z in d7.zone.unique())
print('GHG і ENSO однакові в усіх зонах:', glob_ok)
rows = []
for c, e in LINKS:
    is_global = (c in STAGES['GHG'] and e in STAGES['Океанічні цикли'])
    for z, g in ([('глобально', d7[d7.zone == 'UA_main'])] if is_global else list(d7.groupby('zone'))):
        for k in LAGS:
            r, p, n, ne = corr_neff(g[c].shift(k).values, g[e].values)
            rows.append(dict(cause=c, effect=e, zone=z, lag=k, r=r, p=p, n=n, n_eff=ne))
t7 = pd.DataFrame(rows); t7['p_fdr'] = bh_fdr(t7.p)
t7['sig'] = (t7.p_fdr < .05)
t7.round(4).to_csv(f'{OUT_DIR}/chain_links_07_all_lags.csv', index=False)
# найкращий лаг по ланці/зоні серед значущих (за |r|)
best = (t7[t7.sig].assign(ar=lambda x: x.r.abs()).sort_values('ar', ascending=False).groupby(['cause', 'effect', 'zone']).head(1)
        .drop(columns='ar').sort_values(['cause', 'effect', 'zone']))
best.round(4).to_csv(f'{OUT_DIR}/chain_links_07_best.csv', index=False)
print(f'{len(t7)} тестів, значущих після FDR: {int(t7.sig.sum())}; ланок зі значущим зв\'язком хоча б в одній зоні: {best.groupby(["cause","effect"]).ngroups} з {len(LINKS)}')
print(best[['cause', 'effect', 'zone', 'lag', 'r', 'p_fdr']].round(3).to_string(index=False))

In [ ]:
ZC = {'Австралія': ['AU_control', 'AU_main'], 'Україна': ['UA_control', 'UA_main', 'UA_second']}
edges = {}
for ctry, zs in ZC.items():
    E = []
    for c, e in LINKS:
        sub = best[(best.cause == c) & (best.effect == e)]
        sub = sub[sub.zone.isin(zs + ['глобально'])]
        if sub.empty: continue
        need = 1 if (sub.zone == 'глобально').any() else int(np.ceil(len(zs) / 2 + .01))
        if len(sub) >= need and np.sign(sub.r).nunique() == 1:
            E.append((c, e, int(sub.lag.median()), sub.r.median()))
    edges[ctry] = E
POS = {}
for i, (stg, vs) in enumerate(STAGES.items()):
    for j, v in enumerate(vs): POS[v] = (i, (len(vs) - 1) / 2 - j)
fig, axs = plt.subplots(1, 2, figsize=(17, 6))
for ax, (ctry, E) in zip(axs, edges.items()):
    for v, (x, y) in POS.items(): ax.text(x, y, v.replace('_anom', '').replace('_lag0', ''), ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', fc='#eef'))
    for c, e, k, r in E:
        (x0, y0), (x1, y1) = POS[c], POS[e]
        ax.annotate('', xy=(x1 - .16, y1), xytext=(x0 + .16, y0), arrowprops=dict(arrowstyle='->', color='tab:green' if r > 0 else 'tab:red', lw=1 + 5 * abs(r), alpha=.7))
        ax.text((x0 + x1) / 2, (y0 + y1) / 2 + .05, f'k={k}, r={r:.2f}', fontsize=6.5, ha='center')
    for i, s in enumerate(STAGES): ax.text(i, 2.3, s, ha='center', weight='bold')
    ax.set_xlim(-.5, 4.5); ax.set_ylim(-1.8, 2.6); ax.axis('off'); ax.set_title(f'{ctry}: значущі ланки (зелене = +, червоне = −; k = лаг у місяцях)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/chain_graph_07.png', dpi=130); plt.show()
pd.DataFrame([dict(country=k, cause=c, effect=e, lag_months=l, r=r) for k, E in edges.items() for c, e, l, r in E]).round(3).to_csv(f'{OUT_DIR}/chain_graph_edges_07.csv', index=False)
for k, E in edges.items(): print(k, len(E), 'ребер:', [(c.split('_')[0], e.split('_')[0], l, round(r, 2)) for c, e, l, r in E])

## Файл 09: різниця зрошувана − контрольна зона (Україна проти Австралії)
Що рахуємо: (A) середні `gap_*` по фазах (до війни / війна / війна+дамба) лише у вегетаційні місяці (`in_season`), із block-bootstrap ДІ;
(B) ефект дамби як різниця-різниць: зміна gap у UA_main (під дамбою) мінус зміна gap у UA_second (без дамби);
(C) тренд річного gap (Манн-Кендалл + Сен); (D) «буфер» посухи: нахил gap_ndvi до аномалії NDVI контрольної зони (від'ємний нахил = зрошення гасить посуху).

In [ ]:
d9 = pd.read_csv(P('09_irrigation_gap_ua_vs_au'), parse_dates=['date']).sort_values(['pair', 'date'])
d9['main'] = d9.pair.str.split('-').str[0]
d9 = d9.merge(d1[['zone', 'date', 'in_season']].rename(columns={'zone': 'main'}), on=['main', 'date'], how='left')
# фази за датою (у UA_second прапорця post_dam немає, бо дамби там нема; але календарний період «після 2023-07» потрібен для різниці-різниць)
d9['phase'] = np.where(d9.post_war == 0, '0_pre_war', np.where(d9.date < '2023-07-01', '1_war', '2_war_after_2023-07'))
GV = ['gap_ndvi', 'gap_sm_root_mm', 'gap_et_mm', 'gap_precip_mm', 'gap_tair_c', 'gap_vpd_kpa']
s9 = d9[d9.in_season == 1]
rows = []
for (pr_, ph), g in s9.groupby(['pair', 'phase']):
    for v in GV:
        lo, hi = boot_mean_ci(g[v].values); rows.append(dict(pair=pr_, phase=ph, variable=v, n=g[v].notna().sum(), mean_gap=g[v].mean(), ci95_low=lo, ci95_high=hi))
tA = pd.DataFrame(rows); tA['excludes_0'] = (tA.ci95_low > 0) | (tA.ci95_high < 0)
print('A. Середній gap у вегетаційні місяці:'); print(tA[tA.variable.isin(['gap_ndvi', 'gap_sm_root_mm', 'gap_et_mm'])].round(3).to_string(index=False))
# B. різниця-різниць дамба
rows = []
for v in ['gap_ndvi', 'gap_sm_root_mm', 'gap_et_mm']:
    ch = {}
    for pr_ in ['UA_main-UA_control', 'UA_second-UA_control']:
        g = s9[s9.pair == pr_]; ch[pr_] = (g[g.phase == '2_war_after_2023-07'][v].values, g[g.phase == '1_war'][v].values)
    # зміна в main
    a1, a0 = ch['UA_main-UA_control']; b1, b0 = ch['UA_second-UA_control']
    d_main = np.nanmean(a1) - np.nanmean(a0); d_sec = np.nanmean(b1) - np.nanmean(b0)
    rng = np.random.default_rng(3); B = 3000
    def bm(x): x = x[~np.isnan(x)]; return x[block_idx(len(x), 6, B, rng)].mean(1)
    dd = (bm(a1) - bm(a0)) - (bm(b1) - bm(b0))
    rows.append(dict(variable=v, change_UA_main=d_main, change_UA_second=d_sec, DiD_dam_effect=d_main - d_sec, ci95_low=np.percentile(dd, 2.5), ci95_high=np.percentile(dd, 97.5),
                     significant=(np.percentile(dd, 2.5) > 0) or (np.percentile(dd, 97.5) < 0)))
tB = pd.DataFrame(rows); print('\nB. Ефект дамби (різниця-різниць UA_main vs UA_second, період після 2023-07 проти війни до 2023-07):'); print(tB.round(3).to_string(index=False))
# C. тренди річного gap
s9y = s9.assign(year=s9.date.dt.year).groupby(['pair', 'year'])[GV].mean().reset_index()
rows = []
for pr_, g in s9y.groupby('pair'):
    for v in GV:
        r = mk_test(g[v].values); p2, _ = mk_tfpw(g[v].values)
        rows.append(dict(pair=pr_, variable=v, sen_per_decade=r['sen'] * 10, tau=r['tau'], p_mk=r['p'], p_prewhitened=p2))
tC = pd.DataFrame(rows); tC['p_use'] = np.fmax(tC.p_mk, tC.p_prewhitened.fillna(tC.p_mk)); tC['p_fdr'] = bh_fdr(tC.p_use)
print('\nC. Тренди річного gap:'); print(tC.round(4).to_string(index=False))
# D. буфер посухи
d9['ctrl_ndvi_anom'] = d9.ndvi_ctrl - d9.groupby(['pair', d9.date.dt.month]).ndvi_ctrl.transform('mean')   # сезонна аномалія NDVI контролю
rows = []
for pr_, g in d9[(d9.in_season == 1)].groupby('pair'):
    g = g[['ctrl_ndvi_anom', 'gap_ndvi']].dropna().reset_index(drop=True); x, y = g.ctrl_ndvi_anom.values, g.gap_ndvi.values
    sl = np.polyfit(x, y, 1)[0]; rng = np.random.default_rng(5); ix = block_idx(len(x), 6, 2000, rng)
    bs = [np.polyfit(x[i], y[i], 1)[0] for i in ix]
    dry = y[x < np.quantile(x, .2)].mean(); norm_ = y[x >= np.quantile(x, .2)].mean()
    rows.append(dict(pair=pr_, slope_gap_vs_ctrl_anom=sl, ci95_low=np.percentile(bs, 2.5), ci95_high=np.percentile(bs, 97.5), mean_gap_driest20pct=dry, mean_gap_other=norm_, n=len(x)))
tD = pd.DataFrame(rows); print('\nD. Буфер посухи (нахил gap_ndvi до аномалії NDVI контролю; <0 = зрошення компенсує посуху):'); print(tD.round(3).to_string(index=False))
with pd.ExcelWriter(f'{OUT_DIR}/irrigation_gap_09.xlsx') as xw:
    for nm, t in [('A_phase_means', tA), ('B_dam_DiD', tB), ('C_trends', tC), ('D_drought_buffer', tD)]: t.round(4).to_excel(xw, sheet_name=nm, index=False)

## Файл 11: парникові ггази та індекси ENSO / NAO / SAM
Тренди (Сен, на рік і на десятиліття), прискорення (чи росте швидкість зростання), зв'язок «швидкість росту CO₂ ↔ Ніньо-3.4» з лагами 0–12 міс. після прибирання тренду та сезонності, часті фази ENSO.

In [ ]:
d11 = pd.read_csv(P('11_greenhouse_gases_climate_indices_monthly'), parse_dates=['date']).sort_values('date').reset_index(drop=True)
GG = ['co2_mlo_ppm', 'co2_global_ppm', 'ch4_global_ppb', 'n2o_global_ppb', 'sf6_global_ppt']
IDX = ['nino34_anom_c', 'nao_index', 'sam_aao_index']
rows = []
for v in GG + IDX:
    r = mk_test(d11[v].values); ann = d11.groupby(d11.date.dt.year)[v].mean()
    rows.append(dict(variable=v, start=d11[v].iloc[:12].mean(), end=d11[v].iloc[-12:].mean(), sen_per_year=r['sen'] * 12, sen_per_decade=r['sen'] * 120,
                     total_change_pct=100 * (d11[v].iloc[-12:].mean() / d11[v].iloc[:12].mean() - 1) if v in GG else np.nan, p_mk=r['p'] if v in GG else mk_test(ann.values)['p']))
t11 = pd.DataFrame(rows)
print('Тренди (для індексів p за річними середніми):'); print(t11.round(4).to_string(index=False))
# швидкість росту (12-місячна різниця) і прискорення
growth = pd.DataFrame({'year': d11.date.dt.year})
for v in GG: growth[v + '_growth'] = d11[v].diff(12)
gy = growth.groupby('year').mean().iloc[1:]; gy.round(3).to_csv(f'{OUT_DIR}/ghg_annual_growth_11.csv')
acc = []
for c in gy.columns:
    r = mk_test(gy[c].values); acc.append(dict(variable=c, mean_growth_first_half=gy[c].iloc[:len(gy) // 2].mean(), mean_growth_second_half=gy[c].iloc[len(gy) // 2:].mean(),
                                               accel_per_decade=r['sen'] * 10, p_mk=r['p']))
tacc = pd.DataFrame(acc); print('\nПрискорення росту (Сен по річних темпах):'); print(tacc.round(4).to_string(index=False))
# CO2 growth vs Nino3.4
g = d11.co2_global_ppm.diff(12); nino = d11.nino34_anom_c
mm = g.index
def detr(s):
    t = np.arange(len(s)); ok = ~s.isna(); k = np.polyfit(t[ok], s[ok], 2); return s - np.polyval(k, t)
gd = detr(g); rows = []
for k in range(0, 13):
    r, p, n, ne = corr_neff(nino.shift(k).values, gd.values); rows.append(dict(nino_leads_months=k, r=r, p=p, n_eff=ne))
tn = pd.DataFrame(rows); tn['p_fdr'] = bh_fdr(tn.p); print('\nШвидкість росту CO₂ (детренд.) vs Ніньо-3.4 (Ніньо веде на k міс.):'); print(tn.round(3).to_string(index=False))
# фази ENSO
d11['phase'] = np.where(d11.nino34_anom_c >= .5, 'El Niño', np.where(d11.nino34_anom_c <= -.5, 'La Niña', 'нейтральна'))
ph = d11.groupby([d11.date.dt.year, 'phase']).size().unstack(fill_value=0); print('\nМісяців у фазі ENSO по роках:'); print(ph.T.to_string())
with pd.ExcelWriter(f'{OUT_DIR}/ghg_climate_11.xlsx') as xw:
    t11.round(4).to_excel(xw, sheet_name='trends', index=False); tacc.round(4).to_excel(xw, sheet_name='acceleration', index=False)
    tn.round(4).to_excel(xw, sheet_name='co2growth_vs_nino34', index=False); ph.to_excel(xw, sheet_name='enso_phase_months')
fig, axs = plt.subplots(2, 2, figsize=(13, 8))
axs[0, 0].plot(d11.date, d11.co2_global_ppm); axs[0, 0].set_title('CO₂ global, ppm')
axs[0, 1].plot(d11.date, d11.ch4_global_ppb, c='tab:orange'); axs[0, 1].set_title('CH₄, ppb')
axs[1, 0].plot(d11.date, g, label='швидкість росту CO₂ (ppm/рік)'); ax2 = axs[1, 0].twinx(); ax2.plot(d11.date, nino, c='r', alpha=.5); axs[1, 0].set_title('Ріст CO₂ (синє) і Ніньо-3.4 (червоне)')
axs[1, 1].bar(tn.nino_leads_months, tn.r); axs[1, 1].set_title('Кореляція росту CO₂ з Ніньо-3.4 по лагу (міс.)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/ghg_11.png', dpi=120); plt.show()

## Короткі висновки по частині 2 (на даних, що були на момент запуску)
* **06 (тренди):** із 50 тестів p<0.05 лише у 5 (очікувана кількість випадкових ≈2.5), після FDR значущих немає. NDVI і опади трендів не мають. Найвиразніший сигнал: спад вологості кореневого шару в Україні (≈ −8 мм/десятиліття, попередньо). При 23 роках потужність тесту невелика, тож «тренду немає» ≠ «стабільно».
* **07 (ланцюг):** сильні ланки «дощ (SPI3) → вологість ґрунту → NDVI» (r ≈ 0.65–0.8 в Австралії, 0.3–0.65 в Україні) і «евапотранспірація → NDVI». Верхні ланки слабкі: ENSO → температура (лаг 3–4 міс., r ≈ 0.22–0.29), SAM → SPI3 в Австралії (лаг 1, r ≈ 0.29). Зв'язок CH₄ → SAM (r = 0.25) імовірно випадковий або трендовий: фізичного механізму на таких часових масштабах немає.
* **09 (зрошення):** gap_ndvi стабільно від'ємний (зрошувані зони в сезон «менш зелені» за контроль). Нахил gap до аномалії NDVI контролю від'ємний в усіх парах, тобто зрошення частково згладжує посуху. Ефект дамби за різницею-різниць: NDVI −0.067 [−0.092; −0.034], ET −6.1 мм, вологість ґрунту незначуща. Фаз війни і дамби мало місяців (8 і 12 сезонних), тому ДІ оптимістичні.
* **11 (GHG):** CO₂ +2.3 ppm/рік, CH₄ +7.7 ppb/рік, N₂O +0.97 ppb/рік, SF₆ +0.32 ppt/рік; швидкість росту всіх чотирьох зростає (CO₂ ≈ +0.35 ppm/рік за десятиліття, CH₄ ≈ 2.5× швидше у другій половині). Швидкість росту CO₂ відстає від Ніньо-3.4 на ≈ 8–12 міс. (r ≈ 0.6–0.65), але ефективна вибірка лише ≈ 17.